# Project 21: Online Retail Customer Purchase Analysis

Uses **pandas** and **NumPy** only.

**How to run:** Runtime → Run all, then choose your dataset CSV when the upload box appears (Step 0, Option A). Prefer Google Drive? Use Option B instead and skip A.

## Step 0: Load the dataset into Colab

**Option A: upload from your computer**

In [ ]:
from google.colab import files
import glob

uploaded = files.upload()   # choose online_retail_customer_purchase_dataset...csv
CSV_PATH = list(uploaded.keys())[0]
print('Using file:', CSV_PATH)

**Option B: read from Google Drive** (skip Option A; edit the path to where you saved the CSV)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/online_retail_customer_purchase_dataset__3_.csv'
# print('Using file:', CSV_PATH)

## Imports and CONFIG

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ CONFIG
FILE = CSV_PATH   # set by the Step 0 cell
COL_ORDER_ID = "Order_ID"
COL_DATE = "Order_Date"
COL_PRODUCT = "Product_Name"
COL_CATEGORY = "Category"
COL_QTY = "Quantity"
COL_PRICE = "Unit_Price_INR"
COL_PAY = "Payment_Method"

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

## 1. Load and inspect

In [ ]:
# ------------------------------------------------ 1. Load and inspect
df = pd.read_csv(FILE)
print("=== 1. DATA OVERVIEW ===")
print("Shape (rows, columns):", df.shape)
print("Columns:", list(df.columns))
print(df.dtypes, "\n")
print(df.head(), "\n")

## 2. Missing values & duplicates

In [ ]:
# ------------------------------------------------ 2. Missing values & duplicates
print("=== 2. MISSING VALUES & DUPLICATES ===")
print("Missing values per column:\n", df.isnull().sum(), "\n")
dup_ids = df[df.duplicated(subset=COL_ORDER_ID, keep=False)]
print("Rows sharing a duplicate Order_ID:", len(dup_ids))
print("Fully duplicated rows:", df.duplicated().sum(), "\n")
df = df.drop_duplicates(subset=COL_ORDER_ID, keep="first").copy()

## 3. Handle missing Quantity / Unit_Price

In [ ]:
# ------------------------------------------------ 3. Handle missing Quantity / Unit_Price
# Strategy: this dataset has no missing values, but the code still handles them:
#  - Unit_Price: median price of the same product (fallback: category, then overall).
#    A product's price is stable, and the median is robust to outliers.
#  - Quantity: median of the product's category (fallback: overall median).
print("=== 3. HANDLING MISSING VALUES ===")
df[COL_QTY] = pd.to_numeric(df[COL_QTY], errors="coerce")
df[COL_PRICE] = pd.to_numeric(df[COL_PRICE], errors="coerce")
n_missing = int(df[[COL_QTY, COL_PRICE]].isnull().sum().sum())

df[COL_PRICE] = df[COL_PRICE].fillna(df.groupby(COL_PRODUCT)[COL_PRICE].transform("median"))
df[COL_PRICE] = df[COL_PRICE].fillna(df.groupby(COL_CATEGORY)[COL_PRICE].transform("median"))
df[COL_PRICE] = df[COL_PRICE].fillna(df[COL_PRICE].median())
df[COL_QTY] = df[COL_QTY].fillna(df.groupby(COL_CATEGORY)[COL_QTY].transform("median"))
df[COL_QTY] = df[COL_QTY].fillna(df[COL_QTY].median()).round().astype(int)

df = df[(df[COL_QTY] > 0) & (df[COL_PRICE] > 0)]
print(f"Missing Quantity/Unit_Price values found and filled: {n_missing}\n")

## 4. Dates

In [ ]:
# ------------------------------------------------ 4. Dates
print("=== 4. DATE FEATURES ===")
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
df = df.dropna(subset=[COL_DATE]).copy()
df["Year"] = df[COL_DATE].dt.year
df["Month"] = df[COL_DATE].dt.month
df["Month_Name"] = df[COL_DATE].dt.month_name()
df["Weekday"] = df[COL_DATE].dt.day_name()
print(df[[COL_DATE, "Year", "Month", "Month_Name", "Weekday"]].head(), "\n")

## 6. Order_Value (rowwise)

In [ ]:
# ------------------------------------------------ 6. Order_Value (row-wise)
# (computed before task 5 so revenue can use it)
df["Order_Value"] = df[COL_QTY] * df[COL_PRICE]

## 5. Totals

In [ ]:
# ------------------------------------------------ 5. Totals
print("=== 5. OVERALL TOTALS ===")
total_orders = df[COL_ORDER_ID].nunique()
total_units = int(df[COL_QTY].sum())
total_revenue = df["Order_Value"].sum()
print(f"Total orders       : {total_orders:,}")
print(f"Total units sold   : {total_units:,}")
print(f"Total sales revenue: INR {total_revenue:,.2f}  (Quantity x Unit_Price_INR)\n")

## 7. Top 10 products by revenue

In [ ]:
# ------------------------------------------------ 7. Top 10 products by revenue
print("=== 7. TOP 10 PRODUCTS BY REVENUE ===")
top_products = (df.groupby(COL_PRODUCT)
                  .agg(Units_Sold=(COL_QTY, "sum"), Revenue=("Order_Value", "sum"))
                  .sort_values("Revenue", ascending=False)
                  .head(10))
print(top_products, "\n")

## 8. Category ranking

In [ ]:
# ------------------------------------------------ 8. Category ranking
print("=== 8. CATEGORY RANKING ===")
cat_summary = (df.groupby(COL_CATEGORY)
                 .agg(Total_Quantity=(COL_QTY, "sum"), Total_Revenue=("Order_Value", "sum")))
cat_summary["Qty_Rank"] = cat_summary["Total_Quantity"].rank(ascending=False, method="min").astype(int)
cat_summary["Revenue_Rank"] = cat_summary["Total_Revenue"].rank(ascending=False, method="min").astype(int)
cat_summary = cat_summary.sort_values("Total_Revenue", ascending=False)
print(cat_summary, "\n")

## 9. Average order value by payment method

In [ ]:
# ------------------------------------------------ 9. Average order value by payment method
print("=== 9. AVERAGE ORDER VALUE BY PAYMENT METHOD ===")
pay_avg = (df.groupby(COL_PAY)["Order_Value"]
             .agg(Avg_Order_Value="mean", Orders="count")
             .sort_values("Avg_Order_Value", ascending=False))
print(pay_avg.round(2), "\n")

## 10. Busiest days & peak month

In [ ]:
# ------------------------------------------------ 10. Busiest days & peak month
print("=== 10. BUSIEST DAYS & PEAK MONTH ===")
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_summary = (df.groupby("Weekday")
                     .agg(Orders=(COL_ORDER_ID, "nunique"), Revenue=("Order_Value", "sum"))
                     .reindex(weekday_order))
print("Orders by weekday:\n", weekday_summary, "\n")
busiest_weekday = weekday_summary["Orders"].idxmax()
print("Busiest weekday (by orders):", busiest_weekday)

top_dates = (df.groupby(df[COL_DATE].dt.date)[COL_ORDER_ID].nunique()
               .sort_values(ascending=False).head(5))
print("Top 5 busiest calendar dates:\n", top_dates, "\n")

monthly = (df.groupby(["Year", "Month"])
             .agg(Orders=(COL_ORDER_ID, "nunique"), Units=(COL_QTY, "sum"), Revenue=("Order_Value", "sum"))
             .reset_index())
print("Monthly sales trend:\n", monthly, "\n")
peak = monthly.loc[monthly["Revenue"].idxmax()]
peak_label = f"{int(peak['Year'])}-{int(peak['Month']):02d}"
print(f"Peak month (by revenue): {peak_label} (INR {peak['Revenue']:,.2f})\n")

## 11. Dictionary & set

In [ ]:
# ------------------------------------------------ 11. Dictionary & set
print("=== 11. DICTIONARY & SET ===")
sales_by_category = df.groupby(COL_CATEGORY)["Order_Value"].sum().round(2).to_dict()
unique_categories = set(df[COL_CATEGORY].unique())
print("Sales by category (dict):", sales_by_category)
print("Unique categories (set):", unique_categories, "\n")

## 12. Function

In [ ]:
# ------------------------------------------------ 12. Function
def get_category_sales(category_name, data=df):
    """Return total sales for a category from the cleaned data (case-insensitive)."""
    mask = data[COL_CATEGORY].str.lower() == str(category_name).lower()
    if not mask.any():
        return f"Category '{category_name}' not found. Available: {sorted(data[COL_CATEGORY].unique())}"
    return round(data.loc[mask, "Order_Value"].sum(), 2)

print("=== 12. FUNCTION DEMO ===")
for cat in sorted(unique_categories):
    print(f"get_category_sales('{cat}') -> {get_category_sales(cat):,.2f}")
print(get_category_sales("Toys"), "\n")

## 13. NumPy statistics

In [ ]:
# ------------------------------------------------ 13. NumPy statistics
print("=== 13. ORDER VALUE STATISTICS (NumPy) ===")
values = df["Order_Value"].to_numpy()
mean_v, median_v, std_v = np.mean(values), np.median(values), np.std(values)
print(f"Mean   : {mean_v:,.2f}")
print(f"Median : {median_v:,.2f}")
print(f"Std Dev: {std_v:,.2f}\n")

## 14. IQR outliers & bands

In [ ]:
# ------------------------------------------------ 14. IQR outliers & bands
print("=== 14. OUTLIERS (IQR) & ORDER-VALUE BANDS ===")
q1, q3 = np.percentile(values, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:,.2f}  Q3={q3:,.2f}  IQR={iqr:,.2f}  Bounds=({lower:,.2f}, {upper:,.2f})")

outliers = df[(df["Order_Value"] < lower) | (df["Order_Value"] > upper)]
print(f"Unusual orders (outside IQR bounds): {len(outliers)}")
print(outliers[[COL_ORDER_ID, COL_PRODUCT, COL_QTY, COL_PRICE, "Order_Value"]]
      .sort_values("Order_Value", ascending=False).head(15), "\n")

# Bands: Low (<= Q1), Medium (Q1 to Q3), High (> Q3)
df["Value_Band"] = pd.cut(df["Order_Value"], bins=[-np.inf, q1, q3, np.inf],
                          labels=["Low", "Medium", "High"])
print("Order-value band counts:\n", df["Value_Band"].value_counts(), "\n")

## 15. Product class & recommendations

In [ ]:
# ------------------------------------------------ 15. Product class & recommendations
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __repr__(self):
        return f"Product(name='{self.name}', price={self.price:,.2f})"

print("=== 15. PRODUCT CLASS ===")
prices = df.groupby(COL_PRODUCT)[COL_PRICE].mean()
top5_products = [Product(n, prices[n]) for n in top_products.index[:5]]
for p in top5_products:
    print(p)
print()

best_cat, worst_cat = cat_summary.index[0], cat_summary.index[-1]
best_pay = pay_avg.index[0]
low_products = (df.groupby(COL_PRODUCT)["Order_Value"].sum().sort_values().head(3).index.tolist())

print("=== FIVE SALES RECOMMENDATIONS ===")
recommendations = [
    f"1. Prioritise stock and ads for top earners like '{top_products.index[0]}' and "
    f"'{top_products.index[1]}', and for the '{best_cat}' category, which brings in the most revenue.",
    f"2. Lift the weakest category ('{worst_cat}') and low-revenue products "
    f"({', '.join(low_products)}) with bundles, discounts or better visibility.",
    f"3. Nudge customers toward '{best_pay}' (e.g., small cashback) since it has the highest average order value.",
    f"4. Plan campaigns and staffing around the peak month ({peak_label}) and busiest weekday "
    f"({busiest_weekday}); repeat seasonal offers that worked.",
    f"5. Set free-shipping or bundle thresholds just above the median order value (INR {median_v:,.0f}) "
    f"to move 'Low' band customers up, and reward 'High' band and outlier buyers with loyalty perks.",
]
for r in recommendations:
    print(r)

## Save outputs

In [ ]:
# ------------------------------------------------ Save outputs
df.to_csv("cleaned_retail_orders.csv", index=False)
top_products.to_csv("top_10_products.csv")
cat_summary.to_csv("category_summary.csv")
monthly.to_csv("monthly_sales_trend.csv", index=False)
outliers.to_csv("order_value_outliers.csv", index=False)
print("\nSaved: cleaned_retail_orders.csv, top_10_products.csv, category_summary.csv, "
      "monthly_sales_trend.csv, order_value_outliers.csv")

## Download the output files

In [ ]:
import zipfile
from google.colab import files
out = ['cleaned_retail_orders.csv','top_10_products.csv','category_summary.csv','monthly_sales_trend.csv','order_value_outliers.csv']
with zipfile.ZipFile('retail_outputs.zip','w') as z:
    for f in out: z.write(f)
files.download('retail_outputs.zip')